# 03 - Diversidad de Modelos, Calibración de Probabilidades y Blending

## Curso de Machine Learning: Modelos de Clasificación

Un principio fundamental en el diseño de metamodelos es que **combinar 5 modelos idénticos o altamente correlacionados no aporta ganancia**. Para que un `VotingClassifier` o `StackingClassifier` supere al mejor modelo individual, necesitamos:
1. **Baja correlación entre los errores** de los clasificadores base.
2. **Probabilidades bien calibradas** (`CalibratedClassifierCV`), especialmente cuando combinamos modelos como Naive Bayes o SVM con Regresión Logística.
3. Comprender la alternativa rápida al Stacking en grandes volúmenes de datos: el **Blending (Hold-out Meta-Learning)**.

---

### Contenidos
1. Matriz de Correlación de Predicciones y Diversidad entre Familias de Modelos
2. Calibración de Probabilidades (Platt Scaling e Isotonic Regression) para Soft Voting
3. Implementación de **Blending** con Conjunto de Validación (*Hold-out*) vs *Stacking* (*K-Fold*)
4. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import brier_score_loss, log_loss, accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


In [ ]:
# Analisis de correlacion entre las probabilidades predichas por distintas familias de algoritmos
X_div, y_div = make_classification(n_samples=1500, n_features=20, n_informative=12, class_sep=0.8, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_div, y_div, test_size=0.3, random_state=42, stratify=y_div)

candidatos = {
    'LogReg': make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
    'NaiveBayes': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVC_RBF': make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
}

matriz_probs = {}
for nom, clf in candidatos.items():
    clf.fit(X_tr, y_tr)
    matriz_probs[nom] = clf.predict_proba(X_te)[:, 1]

df_corr = pd.DataFrame(matriz_probs).corr()
plt.figure(figsize=(7, 5.2))
sns.heatmap(df_corr, annot=True, cmap='Blues', fmt='.3f', vmin=0.6, vmax=1.0)
plt.title('Correlacion de Probabilidades entre Modelos Base (Menor = Mayor Diversidad)', fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Calibración de Probabilidades para Mejorar el Soft Voting

Modelos como `GaussianNB` tienden a empujar las probabilidades hacia los extremos ($0$ o $1$) debido a la asunción de independencia, lo que distorsiona el promedio en un `VotingClassifier(voting='soft')`. Aplicando `CalibratedClassifierCV` corregimos esta distorsión y reducimos el **Brier Score** y el **Log-Loss**:


In [ ]:
gnb_crudo = GaussianNB().fit(X_tr, y_tr)
gnb_calibrado = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=5).fit(X_tr, y_tr)

prob_crudo = gnb_crudo.predict_proba(X_te)[:, 1]
prob_cal = gnb_calibrado.predict_proba(X_te)[:, 1]

print(f"GaussianNB Sin Calibrar -> Log-Loss: {log_loss(y_te, prob_crudo):.4f} | Brier Score: {brier_score_loss(y_te, prob_crudo):.4f}")
print(f"GaussianNB Calibrado    -> Log-Loss: {log_loss(y_te, prob_cal):.4f} | Brier Score: {brier_score_loss(y_te, prob_cal):.4f}")

fig, ax = plt.subplots(figsize=(7.5, 5))
CalibrationDisplay.from_estimator(gnb_crudo, X_te, y_te, n_bins=10, name='GaussianNB (Sin Calibrar)', ax=ax)
CalibrationDisplay.from_estimator(gnb_calibrado, X_te, y_te, n_bins=10, name='GaussianNB (Calibrado Isotonico)', ax=ax)
plt.title('Curva de Calibracion de Probabilidades (Reliability Diagram)', fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Blending: La Alternativa Rápida al Stacking

Cuando el dataset tiene millones de registros, ejecutar *5-Fold Cross-Validation* dentro de `StackingClassifier` multiplica por 5 el tiempo de cómputo. El **Blending** divide el entrenamiento en dos subconjuntos disjuntos (`Train_Base` $80\%$ y `Holdout_Blend` $20\%$) o usa `StackingClassifier(..., cv='prefit')` sobre modelos previamente ajustados:


In [ ]:
# Implementacion de Blending rapido (Hold-out 80/20)
X_base, X_blend, y_base, y_blend = train_test_split(X_tr, y_tr, test_size=0.20, random_state=42, stratify=y_tr)

modelos_blend = [
    make_pipeline(StandardScaler(), LogisticRegression(random_state=42)),
    RandomForestClassifier(n_estimators=100, random_state=42),
    GradientBoostingClassifier(n_estimators=100, random_state=42)
]

# 1. Entrenar modelos Nivel 0 en X_base
for m in modelos_blend:
    m.fit(X_base, y_base)

# 2. Generar matriz de meta-features sobre X_blend y X_te
Z_blend = np.column_stack([m.predict_proba(X_blend)[:, 1] for m in modelos_blend])
Z_test = np.column_stack([m.predict_proba(X_te)[:, 1] for m in modelos_blend])

# 3. Ajustar Meta-Modelo en Z_blend
meta_blender = LogisticRegression().fit(Z_blend, y_blend)
preds_blend = meta_blender.predict(Z_test)
print(f"Exactitud de Blending en Test: {accuracy_score(y_te, preds_blend)*100:.2f}%")
